# **Main Notebook: AI School 4: Real-time Computer Vision**

In [ ]:
!pip install ultralytics

Bounding boxes with images

In [ ]:
from IPython.display import Javascript, display
from google.colab.output import eval_js
import cv2
import numpy as np
from ultralytics import YOLO
from google.colab.patches import cv2_imshow
from base64 import b64decode  # Import b64decode function

In [ ]:
# Function to capture webcam photo using JavaScript
def capture_webcam_photo():
    js = Javascript('''
        async function captureImage() {
            const div = document.createElement('div');
            const captureButton = document.createElement('button');
            captureButton.textContent = 'Capture Image';
            div.appendChild(captureButton);

            const video = document.createElement('video');
            div.appendChild(video);
            const stream = await navigator.mediaDevices.getUserMedia({video: true});

            video.srcObject = stream;
            video.style.display = 'block';
            await video.play();

            document.body.appendChild(div);
            await new Promise((resolve) => captureButton.onclick = resolve);

            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);
            stream.getVideoTracks()[0].stop();
            div.remove();
            return canvas.toDataURL('image/jpeg', 0.8);
        }
        captureImage();
    ''')
    display(js)

In [ ]:
# Convert JavaScript image to OpenCV format
def js_to_image(js_data):
    image_bytes = b64decode(js_data.split(',')[1])
    image_np = np.frombuffer(image_bytes, np.uint8)
    return cv2.imdecode(image_np, cv2.IMREAD_COLOR)

# YOLOv8 detection function
def detect_and_display(image, model):
    results = model(image)
    detection_frame = results[0].plot()
    cv2_imshow(detection_frame)

# Load YOLOv8 model
model = YOLO('yolov8n.pt')

In [ ]:
# Main loop to capture images and detect objects
while True:
    print("Click the 'Capture Image' button to take a photo.")
    capture_webcam_photo()
    try:
        js_data = eval_js("captureImage()")
        frame = js_to_image(js_data)
        detect_and_display(frame, model)
    except Exception as e:
        print("Error:", str(e))
        break


Real Time Object Detection

In [ ]:
from IPython.display import Javascript, display
from google.colab.output import eval_js
import numpy as np
import cv2
from base64 import b64decode
from ultralytics import YOLO
from google.colab.patches import cv2_imshow

In [ ]:
# JavaScript to start video stream
def video_stream():
    js = Javascript('''
        var video;
        var div = null;
        var stream;
        var captureCanvas;

        var pendingResolve = null;
        var shutdown = false;

        function removeDom() {
            if (stream) {
                stream.getVideoTracks()[0].stop();
            }
            if (video) {
                video.remove();
            }
            if (div) {
                div.remove();
            }
            video = null;
            div = null;
            stream = null;
            captureCanvas = null;
        }

        function onAnimationFrame() {
            if (!shutdown) {
                window.requestAnimationFrame(onAnimationFrame);
            }
            if (pendingResolve) {
                var result = "";
                if (!shutdown) {
                    captureCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
                    result = captureCanvas.toDataURL('image/jpeg', 0.8);
                }
                var lp = pendingResolve;
                pendingResolve = null;
                lp(result);
            }
        }

        async function createDom() {
            if (div !== null) {
                return;
            }
            div = document.createElement('div');
            div.style.border = '2px solid black';
            div.style.padding = '3px';
            div.style.width = '100%';
            div.style.maxWidth = '640px';
            document.body.appendChild(div);

            video = document.createElement('video');
            video.style.display = 'block';
            video.width = div.clientWidth - 6;
            video.setAttribute('playsinline', '');
            stream = await navigator.mediaDevices.getUserMedia(
                {video: true}
            );
            div.appendChild(video);

            video.srcObject = stream;
            await video.play();

            captureCanvas = document.createElement('canvas');
            captureCanvas.width = 640;
            captureCanvas.height = 480;

            window.requestAnimationFrame(onAnimationFrame);
        }

        async function stream_frame() {
            if (!shutdown) {
                await createDom();
            }
            return await new Promise(function(resolve, reject) {
                pendingResolve = resolve;
            });
        }

        shutdown = false;
        ''')
    display(js)

In [ ]:
# Convert JavaScript image data to OpenCV image
def js_to_image(js_data):
    image_bytes = b64decode(js_data.split(',')[1])
    image_np = np.frombuffer(image_bytes, np.uint8)
    return cv2.imdecode(image_np, cv2.IMREAD_COLOR)

# YOLOv8 detection function
def detect_objects(frame, model):
    results = model(frame)
    return results[0].plot()  # Return frame with bounding boxes

In [ ]:
# Main function to stream video and perform detection
video_stream()
model = YOLO('yolov8n.pt')  # Load YOLOv8 model
print("Press stop or close the stream to terminate.")

try:
    while True:
        js_data = eval_js("stream_frame()")
        if not js_data:
            break
        frame = js_to_image(js_data)
        detected_frame = detect_objects(frame, model)
        cv2_imshow(detected_frame)
except Exception as e:
    print("Error:", str(e))